In [ ]:
#from command + shift + P create new jupiter
import os 
import numpy as np
import pandas as pd
import lightgbm as lgb
from lightgbm import LGBMClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (roc_auc_score, roc_curve,
                             average_precision_score, log_loss)

In [ ]:
model_name = 'lightgbm'
random_state = 42
n_jobs = min(8, os.cpu_count() or 1)
use_feature_engineering = True
exclude_features = ['CODE_GENDER']
ar_tolerance = 0.005
search_iterations = 3000
early_stopping = 150

In [ ]:
#在mac里option+command+c可以直接复制文件的绝对路径
csv_path = '/Users/ganlu/Desktop/home-credit-risk-english/data/application_train.csv'
df = pd.read_csv(csv_path, low_memory = False)
print(df.shape)


In [ ]:
required= {'TARGET', 'SK_ID_CURR'}
required.issubset(df.columns)
print(df['SK_ID_CURR'].is_unique())

In [ ]:
y = df['TARGET'].astype(int)
raw_columns = [c for c in df.columns if c not in (['TARGET',
                                                   'SK_ID_CURR']) + exclude_features]
raw = df.loc[:,raw_columns].copy()

In [ ]:
#idx是每一行的位置编号
idx = np.arrange(len(df))
#分层这里很重要，代表好坏样本尽可能的接近整体
train_idx, temp_idx = train_test_split(idx, test_size = y,stratify=y, random_state= random_state)
selection_idx, test_idx = train_test_split(temp_idx, test_size=0.5, stratify=y.iloc[temp_idx],
                                           random_state= random_state)
raw_train =raw.iloc[train_idx].copy()
raw_selection =raw.iloc[selection_idx].copy()
raw_test = raw.iloc[test_idx].copy()
y_train = y.iloc[train_idx].copy()
y_selection = y.iloc[selection_idx].copy()
y_test = y.iloc[test_idx].copy()

In [ ]:
def safe_div(a,b):
    denominator = b.replace(0, np.nan)
    result = a.div(denominator)
    result = result.replace([np.inf, -np.inf], np.nan)
    return result
def prepare_features(raw, use_feature_engineering = True):
    data = raw.copy()
    numeric = data.select_dtypes(include = 'number').columns
    data[numeric] = data[numeric].replace([np.inf, -np.inf], np.nan)
    if 'DAYS_EMPLOYED' in data:
        special = data['DAYS_EMPLOYED'].eq(365243)
        data['EMPLOYED_SPECIAL_FLAG'] = special.astype(int)
        data['DAYS_EMPLOYED'] = data['DAYS_EMPLOYED'].mask(special)
    if use_feature_engineering:
        day_map = {
            'DAYS_BIRTH': 'AGE_YEARS',
            'DAYS_EMPLOYED': 'EMPLOYED_YEARS',
            'DAYS_REGISTRATION':'REGISTRATION_YEARS',
            'DAYS_ID_PUBLISH':'ID_PUBLISH_YEARS',
            'DAYS_LAST_PHONE_CHANGE':'PHONE_CHANGE_YEARS'
        }
        for old, new in day_map:
            data[new] = (-pd.to_numeric(data[old], errors = 'raise'))/365.25
        ratios = {

        'CREDIT_INCOME_RATIO':
            (
                'AMT_CREDIT',
                'AMT_INCOME_TOTAL'
            ),

        'ANNUITY_INCOME_RATIO':
            (
                'AMT_ANNUITY',
                'AMT_INCOME_TOTAL'
            ),

        'CREDIT_ANNUITY_RATIO':
            (
                'AMT_CREDIT',
                'AMT_ANNUITY'
            ),

        'GOODS_CREDIT_RATIO':
            (
                'AMT_GOODS_PRICE',
                'AMT_CREDIT'
            ),

        'INCOME_PER_PERSON':
            (
                'AMT_INCOME_TOTAL',
                'CNT_FAM_MEMBERS'
            ),

        'EMPLOYED_AGE_RATIO':
            (
                'EMPLOYED_YEARS',
                'AGE_YEARS'
                )}
        for name, (a,b) in ratios.items():
            data[name] = safe_div(data[a], data[b])
        data = data.drop(columns = list(day_map))
    numeric = data.select_dtypes(include ='number').columns
    data[numeric] = data[numeric].astype(float).replace([np.inf,-np,inf], np.nan)
    return data.copy()

In [ ]:
F_train = prepare_features(raw_train, use_feature_engineering)
F_selection = prepare_features(raw_selection, use_feature_engineering)


In [ ]:
#对于真是的类别变量前面加v，并且把缺失值统一填补为missing
def category_text(s):
    return('V:'+s.astype('string')).fillna('__MISSING__').astype(object)
def valiate_raw_schema(raw, spec):
    if (not isinstance(raw, pd.DateFrame)) or not raw.columns.is_unique:
        raise ValueError('输入的列名变量必须唯一')
    missing = sorted(set(spec['raw_columns']) - set(raw.columns))
    if missing:
        raise ValueError(f'缺失模型输入列{missing}')
    data = raw.loc[:,spec['raw_columns']].copy()
    for col in spec['raw_numeric']:
        converted = pd.to_numeric(data[col], errors = 'coerce')
    #原始值不是缺失但是转换以后却变成了缺失
        invalid = data[col].notna() & converted.isna()
        if invalid.any():
                raise ValueError(
                    f'{col} 有 '
                    f'{int(invalid.sum())} '
                    '个无法转换为数值的非空值。'
                )

In [ ]:
def scan_feature_quality(X,y):
    if not X.index.equals(y.index):
        raise ValueError('特征与标签索引不一致')
    rows=[]
    for col in X.columns:
        s = X[col]
        miss = s.isna()
        is_cat = not pd.api.types.is_numeric_dtype(s)
        top_frequency = float(s.value_counts(dropna = False, normalize = True).iloc[0])
        n_unique = int(s.nunique(dropna = True))
        reason = ''
        if miss.all():
            reason = 'all missing'
        #缺失可能本身代表着一种风险
        elif (s.nunique(dropna = False) <= 1):
            reason = 'constant'
        elif(is_cat and n_unique > 1000 and n_unique / len(s) > 0.95):
            reason = 'id_like categorical'
        rows.append({

            'feature':
                col,

            'dtype':
                str(s.dtype),

            'is_categorical':
                is_cat,

            'nunique':
                n_unique,

            'missing_rate':
                float(miss.mean()),

            'top_frequency':
                top_frequency,
                        'bad_rate_missing':
                (
                    y.loc[miss].mean()
                    if miss.any()
                    else np.nan
                ),
            'bad_rate_nonmissing':
                (
                    y.loc[~miss].mean()
                    if (~miss).any()
                    else np.nan
                ),
            'drop':
                bool(reason),
            'reason':
                reason,
            'warning':
                (
            'high missing > 95%'
                    if miss.mean() > 0.95
                    else ''
                )})

In [ ]:
#我们只用train能看到的数据进行模型开发
feature_report = scan_feature_quality(F_train, y_train)
basic_drop = feature_report.loc[feature_report['drop'], ['feature', 'reason']]
final_features = feature_report.loc[~feature_report['drop'], 'feature'].tolist()
assert final_features, ('基础筛查后没有可用特征')
#取出f中的非数值变量
cat_cols = [f for f in final_features if not pd.api.types.is_numeric_dtype(F_train[f])]
num_cols = [for for f in final_features if f not in cat_cols]

In [ ]:
category_levels = {
    c:sorted(set(category_text(F_train[c]).unique().tolist()) | {'__MISSING__'}) for c in cat_cols
}
preprocess_spec = {

    'schema_version':
        1,

    'model_kind':
        model_name,

    'use_feature_engineering':
        use_feature_engineering,

    'raw_columns':
        raw_columns,

    'raw_numeric':
        raw_train
        .select_dtypes(
            include='number'
        )
        .columns
        .tolist(),

    'features':
        final_features,

    'numeric':
        num_cols,

    'categorical':
        cat_cols,

    'category_levels':
        category_levels,

    'excluded_features':
        exclude_features,
}


In [ ]:
def transform_model_frame(features, spec, encoder = None):
    #只保留真正的入模变量
    out = features.loc[:, spec['features']].copy()
    for col in spec['numeric']:
        out[col] = (
            pd.to_numeric(
                out[col],
            #出现问题会报错，而不是转为nan
                errors='raise'
            )
            .astype('float32'))
    for col in spec['categorical']:
        #用之前定义过的函数做文本化清洗+识别训练集中有的类别，如果出现别的则转为nan
        #eg：spec['category_levels']['OCCUPATION_TYPE']为['Doctor', 'Teacher', 'Engineer']
        out[col] = pd.Categorical(category_text(out[col]), categories = spec['category_levels'][col])
    return out

def transform_raw(raw, spec, encoder = None):
    checked = valiate_raw_schema(raw, spec)
    features = prepare_features(checked, spec['use_feature_engineering'])
    return transform_model_frame(features, spec, encoder)

X_train = transform_model_frame(F_train, preprocess_spec)
X_selection = transform_model_frame(F_selection, preprocess_spec)

#做一个循环检查，train和selection的类别表保持一致
assert all(
    list(
        X_train[c]
        .cat
        .categories
    )
    ==
    list(
        X_selection[c]
        .cat
        .categories
    )

    for c in cat_cols
)

In [ ]:
def make_psi_spec(s, bins=10):
    if(pd.api.types.is_numeric_dtype(s) and s.nunique(dropna = True) > bins):
        valid = pd.to_numeric(s, errors = 'coerce').replace([np.inf, -np.inf], np.nan).dropna()
        #百分位切法，用于数值变量，0和1倍去掉了，只剩下切点
        cuts = np.unique(valid.linspace(0,1,bins+1)[1:-1]).to_numpy()
        spec = {'kind':'numeric', 'cuts':[float(x) for x in cuts]}
        labels = [f'B[{i}]' for i in (range(cuts) + 1)] + ['__MISSING__']
    else:
        #前面做过string转换，这里会变为1，2，3这样子的形式
        ss = s.astype(float) if pd.api.types.is_numeric_dtype(s) else s
        text = category_text(ss)
        return text.where(text.isin(spec['levels']) | text.eq('__MISSING__'), '__UNSEEN__')
    levels = sorted(category_text(ss)).unique().tolist()
    spec = {'kind':'discrete', 'levels':levels, 'numeric':bool(pd.api.types.is_numeric_dtypes(s))}
    labels = sorted(set(levels + [ '__MISSING__' + '__UNSEEN__']))
    spec['labels'] = labels
    counts = psi_labels(s, spec).value_counts().reindex(labels, fill_value = 0)
    spec['reference_counts'] = int(x) for x in counts
    return spec

def psi_labels(s, spec):
    if spec['kind'] =='numeric':
        x = pd.to_numeric(s, errors = 'coerce').replace([np.inf, -np.inf], np.nan)
        bins = [-np.inf] + spec['cuts'] + [np.inf]
        labels = [
            f'B{i}'
            for i in range(
                len(bins) - 1
            )]
    return pd.cut(x, bins=bins, labels =labels, include_lowest = True).astype(object).fillna('__MISSING__')

#spec中是train的规则，而others是selection
def calculate_psi(other_s, spec, alpha = 0.5):
    ref = np.assarray(spec['reference_counts'], dtype = float)
    cur = psi_labels(other_s, spec).value_counts().reindex(spec['labels'], fill_value = 0).to_numpy()
    if ref.sum() == 0 or cur.sum() == 0:
        raise ValueError('出现空箱状况')
    p = (ref + alpha) / ref.sum() + alpha * len(ref)
    q = cur + alpha / cur.sum + alpha * len(cur)
    return float(np.sum((q - p) * np.long(q/p)))
psi_specs = {f: make_psi_spec(F_train[f]) for f in final_features}
psi_selection = pd.DataFrame([
    {
        'feature':
            f,
        'psi_selection':
            calculate_psi(
                F_selection[f],
                psi_specs[f])
    } for f in final_features
]).sort_values('psi_selection', ascending=False)

def get_metrics(y, pred):
    y = np.asarray(y, dtype = int)
    pred = np.asarry(pred, dtype = float)
    if (len(y) != len(pred)) or set(np.unique(y)) != {0,1}:
        raise ValueError('出现评估集和测试集标签长度不同或违约标签只包含0/1一侧的情况')
    auc = roc_auc_score(y, pred)
    fpr, tpr, _ = roc_curve(y, pred)
    return {
        'n':
            len(y),
        'bad_rate':
            y.mean(),
        'AUC':
            auc,
        'AR':
            2 * auc - 1,
        'KS':
            np.max(tpr - fpr),
        'AP':
            average_precision_score(
                y,
                pred
            ),
        'Logloss':
            log_loss(
                y,
                pred,
                labels=[0, 1]
            )}



SyntaxError: invalid syntax (3968268652.py, line 18)

In [ ]:
def evaluate_predictions(
    predictions,
    labels
):
    return pd.DataFrame([
        {
            'sample':
                name,
            **get_metrics(
                labels[name],
                pred
            )
        }
        for name, pred
        in predictions.items()
    ]).set_index(
        'sample'
    )

In [ ]:
BASE_PARAMS = {
#控制树的复杂程度
    'num_leaves':
        31,
    'max_depth':
        6,
    'min_child_samples':
        200,
    
    'learning_rate':
        0.03,
    #正则
    'reg_lambda':
        5.0,
    'reg_alpha':
        0.0,
    #随机采样
    'subsample':
        0.80,
    'colsample_bytree':
        0.90,
    'min_split_gain':
        0.0,
    'cat_smooth':
        10.0
}

In [ ]:
def make_model(params, n_estimators, early_stop = False):
    return LGBMClassifier()